# STAGE A2 SEED 42 — CANONICAL GOOGLE COLAB RESUMPTION NOTEBOOK
**Protocol**: Stage A2 Protocol V1.5 (Amendment 12 Locked)  
**Target Seed**: 42  
**Durable Storage**: `/content/drive/MyDrive/Chuyende-stage-a2/runs/HDFS`  
**Execution Model**: Thin caller wrapping committed `scripts/colab_stage_a2_resume_seed42.py`


In [ ]:
# CELL 1 — Mount Google Drive
from google.colab import drive
from pathlib import Path

drive_mount = Path('/content/drive')
drive.mount(str(drive_mount), force_remount=False)
durable_root = Path('/content/drive/MyDrive/Chuyende-stage-a2')
assert durable_root.exists(), f'Durable directory missing: {durable_root}'
print('Google Drive Mounted Successfully:', durable_root)


In [ ]:
# CELL 2 — Clone Approved Repository & Checkout Execution Commit
import os, subprocess, shutil, re
from pathlib import Path

# RUNTIME PLACEHOLDER: The user must supply the approved 40-hex commit SHA
APPROVED_EXECUTION_COMMIT = "<supplied-after-independent-review>"
if APPROVED_EXECUTION_COMMIT == "<supplied-after-independent-review>" or not re.match(r'^[0-9a-fA-F]{40}$', APPROVED_EXECUTION_COMMIT.strip()):
    raise RuntimeError(f"FATAL: APPROVED_EXECUTION_COMMIT required (40-hex SHA). Current value: '{APPROVED_EXECUTION_COMMIT}'")

repo_dir = Path('/content/Research')
repo_url = 'https://github.com/Minhlike/Chuyende.git'

if repo_dir.exists():
    shutil.rmtree(repo_dir)

print(f'Cloning clean repository from {repo_url}...')
subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)

print(f'Detached checkout of approved commit: {APPROVED_EXECUTION_COMMIT}')
subprocess.run(['git', 'checkout', APPROVED_EXECUTION_COMMIT.strip()], cwd=str(repo_dir), check=True)

head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=str(repo_dir), text=True).strip()
assert head == APPROVED_EXECUTION_COMMIT.strip(), f'Commit mismatch: {head} != {APPROVED_EXECUTION_COMMIT}'

print('Installing dependencies...')
subprocess.run(['pip', 'install', '-e', '.'], cwd=str(repo_dir), check=True)
print('Repository Ready at Approved Commit:', head)


In [ ]:
# CELL 3 — Execute Canonical Seed 42 Resume Launcher
import subprocess, sys
from pathlib import Path

resume_script = Path('/content/Research/scripts/colab_stage_a2_resume_seed42.py')
assert resume_script.exists(), f'Resume script missing at {resume_script}'

# Pass --dry-run for pre-flight verification or --execute for real training
cmd = [sys.executable, '-u', str(resume_script), '--execute', '--execution-commit', APPROVED_EXECUTION_COMMIT.strip()]
print('Executing Canonical Resume Launcher:', ' '.join(cmd))
subprocess.run(cmd, check=True)
